In [ ]:
import json
import pandas as pd
from pathlib import Path

In [ ]:
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)

In [ ]:
PROJECT_PATH = Path.cwd()
DATA_PATH = PROJECT_PATH / "data"
MAPPINGS_PATH = PROJECT_PATH / "mappings"
print(PROJECT_PATH)
print(DATA_PATH)
print(MAPPINGS_PATH)

In [ ]:
def investigate_revolut_data():
    df = pd.read_csv(DATA_PATH / "revolut_euro_2020-2026.csv")

    print("Nans per column :")
    print(df.isna().sum().to_dict(), end="\n\n\n")
    
    
    columns = ["Type", "Product", "Currency", "State"]
    for c in columns:
        print("---")
        print(f"'{c}' unique values: ")
        print(df[c].unique(), end="\n\n\n")

    reverted_count = len(df[df["State"] == "REVERTED"])
    nan_balance_count = df["Balance"].isna().sum()

    print("---")
    print("Types: ")
    print(df.dtypes)

    print("---")
    if reverted_count == nan_balance_count:
        print(f"Match: all {reverted_count} REVERTED rows have NaN balance")
    else:
        print(f"Mismatch: {reverted_count} REVERTED vs {nan_balance_count} NaN balances")

    return df

In [ ]:
df = investigate_revolut_data()

In [ ]:
df.tail()

In [ ]:
df[df["Balance"].isna()]

In [ ]:
def clean_revolut_data(df):
    df = df[df["State"] != "REVERTED"]
    df = df.drop(columns=["Product", "Completed Date", "State"])
    df = df.rename(columns={
        "Type": "type",
        "Started Date": "date",
        "Description": "description",
        "Amount": "amount",
        "Fee": "fee",
        "Currency": "currency",
        "Balance": "balance"
    })
    df["date"] = pd.to_datetime(df["date"])
    print("Data frame is cleaned.")
    return df

In [ ]:
df = clean_revolut_data(df)

In [ ]:
def create_descriptions_json(df):
    descriptions = {d: "blank" for d in sorted(df["description"].unique())}

    out_file = PROJECT_PATH / "mappings" / "revolut_euro_descriptions.json"

    with open(out_file, "w", encoding="utf-8") as f:
        json.dump(descriptions, f, ensure_ascii=False, indent=4)

In [ ]:
# create_descriptions_json(df)

In [ ]:
# p2p = df[df["description"].str.contains("Revolut", case=False, na=False)].sort_values("date", ascending=False).reset_index()
p2p = df[
    (df["description"].str.contains("Revolut", case=False, na=False)) &
    (df["amount"] < 0)
].sort_values("date", ascending=False).reset_index()

In [ ]:
p2p.head()

In [ ]:
p2p[p2p["date"] == "2024-05-02 14:50:01"]

In [ ]:
def revolut_to_revolut(df):
    entries = []
    r2r = df[
        (df["description"].str.contains("Revolut", case=False, na=False)) &
        (df["amount"] < 0)
        ].sort_values("date", ascending=False).reset_index()
    
    out_file = PROJECT_PATH / "mappings" / "rtr_mapping.txt"

    with open(out_file, "w", encoding="utf-8") as f:
        for i, row in r2r.iterrows():
            entries.append(
                {
                    "index": i,
                    "date": str(row["date"]),
                    "amount": row["amount"],
                    "new_description": ""
                }
            )
        json.dump(entries, f, ensure_ascii=False, indent=4)

revolut_to_revolut(df)       

In [ ]:
def replace_negative_p2p_descriptions(
        df: pd.DataFrame,
        filepath: Path
):

    new_descriptions = set()
    
    with open(filepath) as f:
        descriptions = json.load(f)

    print(f"Found {len(descriptions)} descriptions that need replacement.")
    
    for d in descriptions:
        new_descriptions.add(d["new_description"])
        condition = (
            (df["description"].str.contains("Revolut", case=False, na=False)) & # broad match intentional — Revolut Ltd is also P2P, one-time use only
            (df["date"] == pd.to_datetime(d["date"])) &
            (df["amount"] == d["amount"])
        )
        df.loc[condition, "description"] = d["new_description"]

    remaining = df[
        df["description"].str.contains("Revolut", case=False, na=False) & df["amount"] < 0
        ].sort_values("date", ascending=False)

    
    
    if remaining.empty:
        print("All NEGATIVE 'Revolut Bank UAB' & 'Revolut Ltd' entries replaced successfully.")
    else:
        print(f"Warning: f{len(remaining)} unresolved rows remain.")

    return df, new_descriptions
    
    

df, new_descriptions = replace_negative_p2p_descriptions(df, PROJECT_PATH / "mappings" / "rtr_mapping_wip.json")

In [ ]:
print(sorted(new_descriptions))

In [ ]:
def replace_positive_p2p_descriptions(df: pd.DataFrame) -> pd.DataFrame:
    
    condition = (
        (df["description"] == "Revolut Bank UAB") &
        (df["amount"] > 0)
    )


    df.loc[condition, "description"] = "P2P: Payback"

    is_empty = df[df["description"].str.contains("Revolut", case=False, na=False)].empty

    if is_empty:
        print("All POSITIVE and NEGATIVE 'Revolut Bank UAB' & 'Revolut Ltd' entries replaced successfully.")
    return df

df = replace_positive_p2p_descriptions(df)

In [ ]:
def create_parquet(df: pd.DataFrame, parquet_name: str):
    out_dir = DATA_PATH / "processed"
    out_dir.mkdir(parents=True, exist_ok=True)
    df.to_parquet(out_dir / parquet_name)

create_parquet(df, "revolut_euro_2020_5-2026.parquet")

In [ ]:
df.head()

In [ ]:
len(df)

In [ ]:
# DES TO rev_euro_wip.JSON | den einai oloklriomeno | oloklirose to kai prosthese to categories column

In [ ]:
# ΕΠΙΣΗΣ ΣΚΕΨΟΥ ΟΤΙ ΕΒΑΛΕΣ ΝΕΕΣ ΚΑΤΗΓΟΡΙΕΣ ΣΤΑ ΜΙΣΚ ΤΟΥ ΡΕΒΟΛΟΥΤ uab αρα θα πρεπει να ξανακανεις ενα json με ολες
# τις κατηγοριες ωστε να γινει επι πλεον μαππινγκ για categories

In [ ]:
def add_category_to_new_descriptions(df, nd):
    file = MAPPINGS_PATH / "llm_euro_rev_categories_wip.json"
    with open(file) as f:
        mapping = json.load(f)

    ['Christina', 'Eirini Tzioka', 'P2P: Fotios Kioutsiokis', 'P2P: Going out', 'P2P: Shopping', 'P2P: Travel']
    
    new_entries = {
        "P2P: Going out": "Going out",
        "P2P: Shopping": "Shopping",
        "P2P: Travel": "Travel",
        "P2P: Home Fotis Kioutsiokis": "Rent",
        "P2P: Payback": "P2P: Payback",
        "Eirini Tzioka": "Phychoanalysis",
        "Christina": "Christina",
    }

    mapping.update(new_entries)

    with open(file, "w", encoding="utf-8") as f:
        json.dump(mapping, f, sort_keys=True, ensure_ascii=False, indent=4)

    return df
    


df = add_category_to_new_descriptions(df, nd=new_descriptions)

In [ ]:
df[df["description"] == "Eirini Tzioka"]

In [ ]:
def create_categories_column(df: pd.DataFrame):
    df = df.sort_values("date").reset_index(drop=True)
    file = MAPPINGS_PATH / "llm_euro_rev_categories_wip.json"
    with open(file) as f:
        mappings = json.load(f)

    df["categories"] = df["description"].map(mappings)

    return df

df = create_categories_column(df)

In [ ]:
df[df["description"] == "Eirini Tzioka"]

In [ ]:
df.columns

In [ ]:
Index(['date', 'description', 'amount', 'balance', 'source', 'category'], dtype='str')
Index(['type', 'date', 'description', 'amount', 'fee', 'currency', 'balance', 'categories'], dtype='str')